<a href="https://colab.research.google.com/github/CaioSilvaUSP/cineticamodelos/blob/master/Analise_estatistica_modelo_Caio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from sklearn.decomposition import PCA
import pandas as pd
import matplotlib.pyplot as plt

def executar_pca(X_scaled, plot_variancia=True):
    """
    Executa PCA e retorna variância explicada e loadings.
    """

    pca = PCA()
    X_pca = pca.fit_transform(X_scaled)

    explained = pca.explained_variance_ratio_

    if plot_variancia:
        plt.figure(figsize=(8,5))
        plt.plot(range(1, len(explained)+1), explained, marker="o")
        plt.xlabel("Componentes principais")
        plt.ylabel("Variância explicada")
        plt.title("Análise de dimensionalidade via PCA")
        plt.grid(True)
        plt.show()

    loadings = pd.DataFrame(
        pca.components_.T,
        columns=[f"PC{i+1}" for i in range(X_scaled.shape[1])],
        index=X_scaled.columns
    )

    return pca, explained, loadings, X_pca


In [2]:
import seaborn as sns

def plotar_heatmap_loadings(loadings, figsize=(16,10)):
    plt.figure(figsize=figsize)
    sns.set(font_scale=1.0)

    sns.heatmap(
        loadings,
        annot=True,
        fmt=".2f",
        center=0,
        linewidths=0.5,
        cbar_kws={"label": "Carga PCA"}
    )

    plt.title("Heatmap de cargas dos componentes principais")
    plt.ylabel("Variáveis")
    plt.xlabel("Componentes principais")
    plt.tight_layout()
    plt.show()


In [3]:
from mpl_toolkits.mplot3d import Axes3D

def plot_3d(df, x_var, y_var, z_var):
    fig = plt.figure(figsize=(10,7))
    ax = fig.add_subplot(111, projection="3d")

    x = df[x_var]
    y = df[y_var]
    z = df[z_var]

    sc = ax.scatter(x, y, z, c=z, s=60)

    ax.set_xlabel(x_var)
    ax.set_ylabel(y_var)
    ax.set_zlabel(z_var)

    plt.title(f"Gráfico 3D de {z_var} em função de {x_var} e {y_var}")
    plt.colorbar(sc, label=z_var)
    plt.show()


In [4]:
from scipy.interpolate import griddata
import numpy as np

def plot_superficie_3d(df, x_var, y_var, z_var, metodo="linear", resolucao=50):
    x = df[x_var].values
    y = df[y_var].values
    z = df[z_var].values

    xi = np.linspace(x.min(), x.max(), resolucao)
    yi = np.linspace(y.min(), y.max(), resolucao)
    XI, YI = np.meshgrid(xi, yi)

    ZI = griddata((x, y), z, (XI, YI), method=metodo)

    fig = plt.figure(figsize=(10,7))
    ax = fig.add_subplot(111, projection="3d")

    surf = ax.plot_surface(XI, YI, ZI, alpha=0.8)
    ax.scatter(x, y, z, s=20)

    ax.set_xlabel(x_var)
    ax.set_ylabel(y_var)
    ax.set_zlabel(z_var)

    plt.title(f"Superfície de {z_var} em função de {x_var} e {y_var}")
    plt.colorbar(surf, label=z_var)
    plt.show()


In [5]:
def plot_superficie_3d_zoom(df, x_var, y_var, z_var, metodo="linear", pmin=5, pmax=95):
    x = df[x_var].values
    y = df[y_var].values
    z = df[z_var].values

    lim_inf = np.percentile(z, pmin)
    lim_sup = np.percentile(z, pmax)

    mask = (z >= lim_inf) & (z <= lim_sup)

    x = x[mask]
    y = y[mask]
    z = z[mask]

    xi = np.linspace(x.min(), x.max(), 50)
    yi = np.linspace(y.min(), y.max(), 50)
    XI, YI = np.meshgrid(xi, yi)

    ZI = griddata((x, y), z, (XI, YI), method=metodo)

    fig = plt.figure(figsize=(10,7))
    ax = fig.add_subplot(111, projection="3d")

    surf = ax.plot_surface(XI, YI, ZI, alpha=0.8)
    ax.scatter(x, y, z, s=20)

    ax.set_xlabel(x_var)
    ax.set_ylabel(y_var)
    ax.set_zlabel(z_var)

    plt.title(f"Superfície 3D com remoção de outliers para {z_var}")
    plt.colorbar(surf, label=z_var)
    plt.show()


In [6]:
def correlacao_pcs_saidas(X_pca, y_dict):
    """
    y_dict exemplo:
    {
        "Keq": y_keq,
        "k_sem_catali": y_k
    }
    """

    df_pcs = pd.DataFrame(X_pca, columns=[f"PC{i+1}" for i in range(X_pca.shape[1])])

    resultados = {}

    for nome_saida, y in y_dict.items():
        resultados[nome_saida] = df_pcs.apply(lambda pc: np.corrcoef(pc, y)[0,1])

    return pd.DataFrame(resultados)


In [7]:
def plotar_correlacoes(correlacoes_df):
    plt.figure(figsize=(10,6))

    for coluna in correlacoes_df.columns:
        plt.plot(correlacoes_df.index, correlacoes_df[coluna], marker="o", label=coluna)

    plt.xticks(rotation=45)
    plt.ylabel("Correlação")
    plt.xlabel("Componentes principais")
    plt.title("Correlação entre PCs e saídas do modelo")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
from scipy.stats import shapiro
import pandas as pd

def teste_normalidade(X):
    resultados = []

    for col in X.columns:
        stat, p = shapiro(X[col].dropna())
        resultados.append({
            "Variavel": col,
            "Estatistica": stat,
            "p_valor": p,
            "Normal_5pct": p > 0.05
        })

    return pd.DataFrame(resultados)


In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan
import statsmodels.api as sm

def teste_homocedasticidade(X, y):
    X_const = sm.add_constant(X)
    modelo = sm.OLS(y, X_const).fit()

    teste = het_breuschpagan(modelo.resid, modelo.model.exog)

    return {
        "Lagrange multiplier": teste[0],
        "p_valor": teste[1],
        "Homocedastico_5pct": teste[1] > 0.05
    }


In [ ]:
from scipy.stats import chi2
import numpy as np

def detectar_outliers_mahalanobis(X, alpha=0.001):
    cov = np.cov(X.values, rowvar=False)
    inv_cov = np.linalg.inv(cov)
    media = X.mean().values

    distancias = []

    for row in X.values:
        diff = row - media
        d = diff.T @ inv_cov @ diff
        distancias.append(d)

    limite = chi2.ppf(1 - alpha, df=X.shape[1])

    return pd.DataFrame({
        "Distancia": distancias,
        "Outlier": np.array(distancias) > limite
    })


In [ ]:
from sklearn.feature_selection import mutual_info_regression

def importancia_mutua(X, y):
    scores = mutual_info_regression(X, y)

    resultado = pd.DataFrame({
        "Variavel": X.columns,
        "Importancia": scores
    }).sort_values("Importancia", ascending=False)

    return resultado


In [ ]:
from sklearn.utils import resample
import numpy as np

def bootstrap_correlacao(X, y, n_iter=500):
    resultados = {col: [] for col in X.columns}

    for _ in range(n_iter):
        Xb, yb = resample(X, y)

        for col in X.columns:
            corr = np.corrcoef(Xb[col], yb)[0,1]
            resultados[col].append(corr)

    resumo = []

    for var, vals in resultados.items():
        resumo.append({
            "Variavel": var,
            "Media": np.mean(vals),
            "IC_2_5": np.percentile(vals, 2.5),
            "IC_97_5": np.percentile(vals, 97.5)
        })

    return pd.DataFrame(resumo).sort_values("Media", ascending=False)


In [ ]:
from sklearn.cluster import KMeans

def clusterizar_experimentos(X_scaled, k=3):
    modelo = KMeans(n_clusters=k, random_state=42)
    clusters = modelo.fit_predict(X_scaled)

    return clusters


In [ ]:
def detectar_variaveis_redundantes(corr_matrix, limite=0.9):
    redundantes = []

    for i in range(len(corr_matrix.columns)):
        for j in range(i):
            if abs(corr_matrix.iloc[i, j]) > limite:
                redundantes.append((
                    corr_matrix.columns[i],
                    corr_matrix.columns[j],
                    corr_matrix.iloc[i, j]
                ))

    return pd.DataFrame(redundantes, columns=["Var1", "Var2", "Correlacao"])


In [ ]:
def diagnostico_estatistico_global(X):
    normalidade = teste_normalidade(X)
    corr = X.corr()
    redundancia = detectar_variaveis_redundantes(corr)

    return {
        "percentual_variaveis_normais": normalidade["Normal_5pct"].mean(),
        "pares_redundantes": len(redundancia),
        "media_correlacao_absoluta": corr.abs().mean().mean()
    }
